In [11]:
pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [12]:
from kaggle_secrets import UserSecretsClient
from IPython.display import Markdown, display
import anthropic

key = UserSecretsClient().get_secret("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=key)

In [13]:
patients = {
    "P003": {
        "procedure": "TPLO",
        "summary": """
Who: Duke, 3-year-old male neutered Great Dane
Why presented: Post-op care following right TPLO. History of bilateral hip
dysplasia and a left TPLO earlier this year.
Changes while here: Very vocal post-op; settled and ate after pain/sedation
protocol adjusted. Became agitated with e-collar, which was removed.
Meds started: Buprenorphine.
Dose changes: Initial protocol insufficient; changed to gabapentin (600 mg),
trazodone (300 mg), and methadone.
Meds stopped: Fentanyl, switched to buprenorphine.
Plan: Discharge this morning. Buprenorphine due 08:00, moved to 09:00 pending
discharge time. IV catheter to be removed before discharge. Owners brought
home-cooked food, in a labeled container in the treatment-room fridge.
Watch: Pain/anxiety prior to discharge.
Pending: none
Code status: Not stated. Owners very dedicated.
""",
        "orders": {
            "medications": [
                {"drug": "Gabapentin", "dose": "600 mg", "route": "by mouth",
                 "frequency": "every 8 hours", "duration": "for 14 days"},
                {"drug": "Trazodone", "dose": "300 mg", "route": "by mouth",
                 "frequency": None, "duration": None},
            ],
            "other": ["Remove IV catheter prior to discharge"],
        },
    }
}

protocols = {
    "TPLO": """
HARBORVIEW VETERINARY SURGERY CENTER (FICTIONAL) - STANDARD TPLO AFTERCARE
Applies unless the doctor specifies otherwise.
ACTIVITY: Weeks 0-2 strict confinement; leash walks for bathroom breaks only,
5-10 minutes, 3-4 times daily. Weeks 2-8 leash walks may increase about
5 minutes per week. No running, jumping, stairs, rough play, or off-leash
activity until cleared at the 8-week recheck.
INCISION: Check twice daily for redness, swelling, discharge, or odor. No
bathing or swimming until cleared. E-collar at all times when unsupervised
until suture removal, unless the doctor documents an alternative.
FOLLOW-UP: Recheck and suture removal in 10-14 days. Radiographs at 8 weeks.
CALL IMMEDIATELY IF: sudden non-weight-bearing on the operated leg; incision
opens, bleeds, or drains; pain not controlled by prescribed medications;
vomiting, diarrhea, or refusal to eat for more than 24 hours.
CONTACT: Clinic (555) 010-0100. After-hours (555) 010-0199.
""",
}

In [14]:
patients["P002"] = {
    "procedure": "PNEUMONIA",
    "summary": """
Who: Olive, 9-year-old female spayed DLH
Why presented: Acute coughing, increased respiratory effort, inappetence.
History of recurrent pleural effusion. Radiographs consistent with aspiration
pneumonia.
Changes while here: Weaned off oxygen, much brighter, eating and drinking well.
Meds started: Transitioned to oral meds: amoxicillin-clavulanate,
enrofloxacin, prednisolone.
Dose changes: none
Meds stopped: Ampicillin-sulbactam, butorphanol.
Plan: Discharge today. Continue oral meds. No IV access (both cephalic
catheters lost, saphenous veins poor); no plan to replace since stable on
orals. Owners may follow up about further workup for the effusion.
Watch: Respiratory effort and appetite.
Pending: none
Code status: not stated
""",
    "orders": {
        "medications": [
            {"drug": "Amoxicillin-clavulanate", "dose": "62.5 mg", "route": "by mouth",
             "frequency": "every 12 hours", "duration": "for 14 days"},
            {"drug": "Enrofloxacin", "dose": "22.7 mg", "route": "by mouth",
             "frequency": "every 24 hours", "duration": "for 14 days"},
            {"drug": "Prednisolone", "dose": "5 mg", "route": "by mouth",
             "frequency": "every 24 hours", "duration": "for 7 days"},
        ],
        "other": [],
    },
}

protocols["PNEUMONIA"] = """
HARBORVIEW VETERINARY SURGERY CENTER (FICTIONAL) - STANDARD PNEUMONIA AFTERCARE
Applies unless the doctor specifies otherwise.
ACTIVITY: Keep indoors and quiet. Limit activity until the recheck.
BREATHING: Twice daily, count breaths per minute while your pet is resting
or asleep. Call if it is more than 40 breaths per minute.
FOLLOW-UP: Recheck exam and chest radiographs in 10-14 days.
CALL IMMEDIATELY IF: labored, rapid, or open-mouth breathing; blue or pale
gums; collapse; not eating for more than 24 hours.
CONTACT: Clinic (555) 010-0100. After-hours (555) 010-0199.
"""

In [15]:
patients["P001"] = {
    "procedure": "TRACHEOSTOMY",
    "summary": """
Who: Rosie, 11-year-old female spayed Boxer
Why presented: Episodic noisy, labored breathing. Diagnosed with a laryngeal
mass (suspect carcinoma) and had a temporary tracheostomy placed after poor
recovery from a biopsy procedure.
Changes while here: One episode of respiratory distress yesterday (panting,
restless, temp 103.8F), improved after a bandage compressing the trach site
was loosened. Overnight up, walking, drinking, ate a small amount. Agitated
with stimulation (owner visits especially), settles with acepromazine.
Meds started: Acepromazine PRN for agitation. Fentanyl CRI restarted yesterday.
Dose changes: none
Meds stopped: Fentanyl CRI briefly discontinued yesterday before restart.
Plan: Fentanyl CRI nearly finished; plan to discontinue and monitor, no oral
pain meds. Owner may visit around 10:00. Team to discuss goals of care
(euthanasia vs. permanent trach). May offer different foods.
Watch: Respiratory status, trach occlusion, distress with agitation.
Pending: none
Code status: Not stated. Euthanasia discussed; owners have elected to proceed.
""",
    "orders": {"medications": [], "other": []},
}

In [16]:
PLACEHOLDER = "[STAFF TO COMPLETE FROM PRESCRIPTION: frequency]"

def format_medication(med):
    line = f"{med['drug']} {med['dose']} {med['route']}"
    line += f" {med['frequency']}" if med["frequency"] else f" {PLACEHOLDER}"
    if med["duration"]:
        line += f" {med['duration']}"
    return line

def get_discharge_orders(patient_id):
    patient = patients.get(patient_id)
    if patient is None:
        return f"No discharge orders on file for {patient_id}."
    orders = patient["orders"]
    med_lines = [f"- {format_medication(m)}" for m in orders["medications"]] or ["- (none)"]
    other_lines = [f"- {o}" for o in orders["other"]] or ["- (none)"]
    text = "MEDICATION LINES (copy each exactly):\n" + "\n".join(med_lines)
    text += "\nOTHER ORDERS:\n" + "\n".join(other_lines)
    return text

def get_clinic_protocol(procedure):
    for key, text in protocols.items():
        if key in procedure.upper():
            return text
    return f"No protocol on file for {procedure}."

def run_tool(name, tool_input):
    if name == "get_discharge_orders":
        return get_discharge_orders(tool_input["patient_id"])
    if name == "get_clinic_protocol":
        return get_clinic_protocol(tool_input["procedure"])
    return f"Unknown tool: {name}"

tools = [
    {
        "name": "get_discharge_orders",
        "description": "Get the doctor's discharge orders for a patient: the exact medication lines and other orders approved by the doctor.",
        "input_schema": {
            "type": "object",
            "properties": {"patient_id": {"type": "string"}},
            "required": ["patient_id"],
        },
    },
    {
        "name": "get_clinic_protocol",
        "description": "Get the clinic's standard aftercare protocol for a procedure, such as TPLO.",
        "input_schema": {
            "type": "object",
            "properties": {"procedure": {"type": "string"}},
            "required": ["procedure"],
        },
    },
]

In [17]:
system_prompt = """You draft discharge instructions for a veterinary hospital.
You do not prescribe or make clinical decisions. You report what is in the
record, in plain language for the owner. Your output is a DRAFT for the
doctor to review and approve.

Before drafting, always call get_discharge_orders for the patient and
get_clinic_protocol for the procedure. The doctor's orders come ONLY from
get_discharge_orders. If a tool finds nothing on file, flag it; never guess.

Sources, in order of authority:
1. The doctor's discharge orders.
2. The clinic protocol (standard aftercare only).
3. Anything else in the case summary (for example, staff workarounds).

Medication rules:
- Copy each medication line from get_discharge_orders into the owner
  instructions EXACTLY as written, one per line, with no bold or other
  formatting inside the line. Do not reword, reorder, or add to them.
- Mention no other drugs anywhere in the owner instructions.
- Add no medication advice that is not in the orders (for example, do not
  say "complete the full course").
- Flag every [STAFF TO COMPLETE FROM PRESCRIPTION] line in the staff review.

Protocol rules:
- If the record is silent on aftercare, use the clinic protocol.
- Include EVERY point from each protocol section. You may simplify the
  wording, but never drop or soften a point's meaning.
- If staff actions conflict with the protocol and no doctor order covers
  it, follow the protocol for the owner and flag the conflict.

Never in the owner instructions:
- Anything not in the doctor's orders or the clinic protocol, including
  suggested alternatives such as recovery suits.
- Internal staff notes, opinions about the owners, or closing pleasantries
  based on them.
- References to other paperwork or sections that do not exist.
- Bold text or other emphasis formatting.

Staff review rules:
- Make the first item a one-line summary of the take-home medications
  from the orders, for example: "Take-home medications: gabapentin,
  trazodone." This is a statement, not a question.
- Pain medication check: if the case summary shows pain medication was
  used in the hospital (for example, opioids such as methadone,
  buprenorphine, or fentanyl) AND the discharge orders contain NO pain
  medication of any kind, flag it, for example: "Patient received
  methadone in hospital; no pain medication is in the discharge orders.
  Please confirm." Gabapentin, NSAIDs, and opioids all count as pain
  medication. If ANY pain medication is in the discharge orders, do not
  mention the in-hospital pain drugs at all.
- Mention code status only if the patient is going to euthanasia.

If the patient is going to euthanasia, write no owner instructions; write
a short staff note of the plan and flag any missing details.

Output exactly two sections:
STAFF REVIEW: a numbered list of every flag, conflict, and protocol default.
OWNER INSTRUCTIONS: clear, plain-language instructions for the owner.
"""

In [18]:
# Drug names to watch for, so code can catch an unordered drug
WATCH_LIST = [
    "gabapentin", "trazodone", "carprofen", "rimadyl", "meloxicam",
    "grapiprant", "galliprant", "buprenorphine", "methadone", "fentanyl",
    "hydromorphone", "tramadol", "acepromazine", "amoxicillin", "clavamox",
    "enrofloxacin", "prednisolone", "maropitant", "cerenia", "cephalexin",
    "ampicillin", "sulbactam", "unasyn", "butorphanol",
]

def normalize(text):
    # lowercase, strip markdown symbols, collapse extra spaces
    for symbol in ["*", "_", "`"]:
        text = text.replace(symbol, "")
    return " ".join(text.lower().split())

def check_medications(draft, orders):
    meds = orders["medications"]
    if "OWNER INSTRUCTIONS" not in draft:
        if meds:
            return ["No OWNER INSTRUCTIONS section, but medications were ordered."]
        return []
    owner = normalize(draft.split("OWNER INSTRUCTIONS", 1)[1])
    ordered_names = [m["drug"].lower() for m in meds]
    problems = []

    # 1. Every ordered medication line must appear exactly
    for med in meds:
        line = normalize(format_medication(med))
        if line not in owner:
            problems.append(f"Missing or altered medication line: '{format_medication(med)}'")

    # 2. No drug may appear that the doctor did not order
    for name in WATCH_LIST:
        if name in owner and not any(name in drug for drug in ordered_names):
            problems.append(f"Unordered drug in owner instructions: {name}")

    return problems

def draft_discharge(patient_id, max_steps=8, max_fixes=1):
    summary = patients[patient_id]["summary"]
    orders = patients[patient_id]["orders"]
    messages = [{
        "role": "user",
        "content": f"Patient ID: {patient_id}\nCase summary:\n{summary}\nDraft discharge instructions.",
    }]
    fixes = 0

    for step in range(max_steps):
        response = client.messages.create(
            model="claude-sonnet-5",
            max_tokens=4000,
            system=system_prompt,
            tools=tools,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"Claude called {block.name} with {block.input}")
                    results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": run_tool(block.name, block.input),
                    })
            messages.append({"role": "user", "content": results})

        elif response.stop_reason == "end_turn":
            draft = "".join(b.text for b in response.content if b.type == "text")
            problems = check_medications(draft, orders)
            if not problems:
                print("Medication check: PASSED")
                return draft
            print("Medication check: FAILED")
            for p in problems:
                print("  -", p)
            if fixes < max_fixes:
                fixes += 1
                print("Sending back to Claude to fix...")
                messages.append({
                    "role": "user",
                    "content": "The medication check failed:\n" + "\n".join(problems)
                               + "\nRewrite the full draft and fix these problems.",
                })
            else:
                return "BLOCKED: medication check failed after a retry. Not safe to use.\n\n" + "\n".join(problems)

        else:
            return f"BLOCKED: draft stopped early ({response.stop_reason}). Not safe to use."

    return "BLOCKED: too many steps without finishing. Not safe to use."

In [19]:
import copy

# Extra test: Duke with gabapentin removed. The pain flag SHOULD appear.
patients["P004"] = copy.deepcopy(patients["P003"])
patients["P004"]["orders"]["medications"] = [
    m for m in patients["P004"]["orders"]["medications"] if m["drug"] != "Gabapentin"
]

for patient_id in ["P001", "P002", "P003", "P004"]:
    print(f"\n========== {patient_id} ==========")
    result = draft_discharge(patient_id)
    display(Markdown(result))


========== P001 ==========
Claude called get_discharge_orders with {'patient_id': 'P001'}
Claude called get_clinic_protocol with {'procedure': 'tracheostomy'}
Medication check: PASSED


STAFF REVIEW:

1. Patient is going to euthanasia: case summary states owners have discussed goals of care and have elected to proceed with euthanasia. Per policy, no owner discharge instructions are being drafted — only this staff note.
2. Discharge orders on file show no medication lines and no other orders — confirm this is correct given the euthanasia plan, and confirm no take-home medications are needed.
3. Code status is not explicitly stated in the record; since the plan is euthanasia, please confirm/document code status and euthanasia scheduling details (timing, consent, owner presence).
4. No clinic protocol is on file for "tracheostomy" — flagging in case aftercare/procedural documentation is needed for the record.
5. Plan notes "team to discuss goals of care (euthanasia vs. permanent trach)" and separately states owners have elected to proceed with euthanasia — please confirm this discussion has concluded and the euthanasia plan is finalized before proceeding.
6. Patient received fentanyl CRI (restarted yesterday, nearly finished) and acepromazine PRN in hospital; plan states no oral pain meds and discharge orders show none — please confirm this is intentional given the euthanasia plan.
7. Owner visit was planned around 10:00 — confirm whether this aligns with euthanasia scheduling/timing.

OWNER INSTRUCTIONS:

Not applicable — patient is going to euthanasia; no owner discharge instructions are provided per policy. Please coordinate directly with the family regarding the euthanasia plan and any remaining questions.


========== P002 ==========
Claude called get_discharge_orders with {'patient_id': 'P002'}
Claude called get_clinic_protocol with {'procedure': 'aspiration pneumonia'}
Medication check: PASSED


STAFF REVIEW:
1. Take-home medications: amoxicillin-clavulanate, enrofloxacin, prednisolone.
2. Patient received butorphanol in hospital; no pain medication is in the discharge orders. Please confirm.
3. No other orders were found in the discharge orders beyond the three medication lines; aftercare below defaults to the standard clinic pneumonia protocol since the record is silent on it.
4. Case summary notes no IV access currently in place (catheters lost, saphenous veins poor) with no plan to replace; this is an internal clinical note and has been omitted from owner instructions, but flagging here in case it should be documented elsewhere.
5. Code status not stated — no action needed since patient is not going to euthanasia.

OWNER INSTRUCTIONS:

Medications:
- Amoxicillin-clavulanate 62.5 mg by mouth every 12 hours for 14 days
- Enrofloxacin 22.7 mg by mouth every 24 hours for 14 days
- Prednisolone 5 mg by mouth every 24 hours for 7 days

Activity:
- Keep Olive indoors and quiet. Limit her activity until her recheck appointment.

Breathing checks:
- Twice a day, count how many breaths Olive takes per minute while she is resting or asleep.
- Call the clinic if her breathing rate is more than 40 breaths per minute.

Follow-up:
- Schedule a recheck exam and chest X-rays in 10–14 days.

Call immediately if you notice any of the following:
- Labored, rapid, or open-mouth breathing
- Blue or pale gums
- Collapse
- Not eating for more than 24 hours

Contact us:
- Clinic: (555) 010-0100
- After-hours: (555) 010-0199


========== P003 ==========
Claude called get_discharge_orders with {'patient_id': 'P003'}
Claude called get_clinic_protocol with {'procedure': 'TPLO'}
Medication check: PASSED


STAFF REVIEW:
1. Take-home medications: gabapentin, trazodone.
2. Trazodone order is missing frequency ([STAFF TO COMPLETE FROM PRESCRIPTION]) — please fill in from the prescription before giving to owner.
3. Gabapentin is included in the discharge orders (gabapentin counts as pain medication), so no flag needed regarding in-hospital buprenorphine/methadone use.
4. E-collar conflict: clinic protocol requires an e-collar at all times when unsupervised until suture removal unless the doctor documents an alternative. Staff removed the e-collar in-house due to agitation, but no doctor order waiving the e-collar is on file. Owner instructions below follow the standard protocol (e-collar required); please confirm whether an exception should be documented.
5. Confirm IV catheter has been removed prior to discharge per order on file.
6. Code status not stated — not applicable for this owner handout since patient is discharging home, not going to euthanasia.

OWNER INSTRUCTIONS:

Medications:
- Gabapentin 600 mg by mouth every 8 hours for 14 days
- Trazodone 300 mg by mouth [STAFF TO COMPLETE FROM PRESCRIPTION: frequency]

Activity restrictions (right TPLO):
- For the first 2 weeks: strict confinement. Only leash walks for bathroom breaks, 5–10 minutes, 3–4 times a day.
- From weeks 2–8: leash walks may gradually increase by about 5 minutes per week.
- No running, jumping, using stairs, rough play, or off-leash activity until Duke is cleared at the 8-week recheck.

Incision care:
- Check the incision twice daily for redness, swelling, discharge, or odor.
- No bathing or swimming until your vet clears it.
- Duke should wear his e-collar at all times when he is not being directly supervised, until his sutures are removed.

Follow-up care:
- Schedule a recheck and suture removal in 10–14 days.
- X-rays are needed at the 8-week mark.

Call the clinic immediately if you notice any of the following:
- Sudden inability or refusal to bear weight on the operated leg.
- The incision opens, bleeds, or drains.
- Pain that is not controlled by the prescribed medications.
- Vomiting, diarrhea, or refusal to eat for more than 24 hours.

Contact us:
- Clinic: (555) 010-0100
- After-hours: (555) 010-0199


========== P004 ==========
Claude called get_discharge_orders with {'patient_id': 'P004'}
Claude called get_clinic_protocol with {'procedure': 'TPLO'}
Medication check: PASSED


STAFF REVIEW:

1. Take-home medications: trazodone.
2. The trazodone line is missing the frequency — flagged as [STAFF TO COMPLETE FROM PRESCRIPTION: frequency]. Please complete before giving to owner.
3. Patient received buprenorphine and methadone in hospital (opioids); no pain medication is in the discharge orders (trazodone is not a pain medication). Please confirm whether pain medication should be added.
4. E-collar was removed in hospital due to agitation, but clinic TPLO protocol requires an e-collar at all times when unsupervised until suture removal unless the doctor documents an alternative. No such doctor documentation is on file — please confirm whether an e-collar order/alternative should be documented, or owner instructions will default to protocol (e-collar required).
5. Confirm IV catheter has been removed prior to discharge per order.
6. No doctor order addresses activity restriction, incision care, or follow-up schedule — owner instructions default to standard TPLO protocol below.
7. Code status not stated; not applicable as patient is not going to euthanasia.

OWNER INSTRUCTIONS:

Medication:
- Trazodone 300 mg by mouth [STAFF TO COMPLETE FROM PRESCRIPTION: frequency]

Activity:
- Weeks 0–2: strict confinement. Leash walks only for bathroom breaks, 5–10 minutes, 3–4 times daily.
- Weeks 2–8: leash walk time may increase by about 5 minutes per week.
- No running, jumping, stairs, rough play, or off-leash activity until cleared at the 8-week recheck.

Incision care:
- Check the incision twice daily for redness, swelling, discharge, or odor.
- No bathing or swimming until cleared by the doctor.
- Duke should wear an e-collar at all times when unsupervised until his sutures are removed.

Follow-up:
- Recheck and suture removal in 10–14 days.
- Radiographs at 8 weeks.

Call the clinic immediately if you notice any of the following:
- Sudden inability or refusal to bear weight on the operated leg.
- The incision opens, bleeds, or drains.
- Pain that is not controlled by the prescribed medication.
- Vomiting, diarrhea, or refusal to eat for more than 24 hours.

Contact: Clinic (555) 010-0100. After-hours (555) 010-0199.

In [20]:
bad_draft = """
OWNER INSTRUCTIONS:
- Gabapentin 60 mg by mouth every 8 hours for 14 days
- Trazodone 300 mg by mouth [STAFF TO COMPLETE FROM PRESCRIPTION: frequency]
- Carprofen 100 mg by mouth every 12 hours
"""
for p in check_medications(bad_draft, patients["P003"]["orders"]):
    print(p)

Missing or altered medication line: 'Gabapentin 600 mg by mouth every 8 hours for 14 days'
Unordered drug in owner instructions: carprofen
